# 带原文引用的证据检索

此演示返回固定语料中的文档摘录和可追溯 ID。排序分数不代表答案正确率；当前没有调用生成式模型。

In [1]:
from pathlib import Path
import json, sys, os
import pandas as pd
from IPython.display import display, Image
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'project/src/ragstudy').is_dir())
for folder in ('.vendor', '.vendor-hf', '.vendor-onnx-compat'):
    path = root / 'project' / folder
    if path.is_dir():
        sys.path.insert(0, str(path))
sys.path.insert(0, str(root / 'project/src'))
artifacts = sorted(path for path in (root / 'project/artifacts').iterdir()
                   if (path / 'results.json').is_file() and (path / 'search_bundle.joblib').is_file())
assert artifacts, 'First run python project/scripts/run_experiment.py --backend lsa'
artifact = Path(os.environ['RAG_ARTIFACT']).resolve() if os.environ.get('RAG_ARTIFACT') else artifacts[-1]
result = json.loads((artifact / 'results.json').read_text(encoding='utf-8'))
print('Actual local run:', artifact.name, '| backend:', result['retrieval_metadata']['semantic']['method'])

Actual local run: frozen_e2324b7a967c5dbe2766ecc3 | backend: frozen_onnx_fp32_cls_cosine


In [2]:
from ragstudy.search import load_bundle, search
from threadpoolctl import threadpool_limits
with threadpool_limits(limits=2):
    evidence = search(load_bundle(artifact), 'How do I resolve an IBM WebSphere connection timeout?', method='logistic', limit=3)
display(pd.DataFrame(evidence['evidence']))
display(evidence['timing_ms'])

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


,citation,document_id,ranking_score,excerpt
0,[swg1PK68853.txt],swg1PK68853.txt,0.143571,Title: IBM PK68853: COM.IBM.CORBA.REQUESTTIMEO...
1,[swg21512329.txt],swg21512329.txt,0.041796,Title: IBM How to change the device communicat...
2,[swg1PK27001.txt],swg1PK27001.txt,0.026107,Title: IBM PK27001: CONNECTION POOL ISSUE WITH...


{'retrieval': 3012.61780000641,
 'candidate_features': 12.076199986040592,
 'rerank_and_output': 0.46300000394694507,
 'total': 3025.1569999963976}